# Ablation функции потерь

Проверяем две гипотезы о текущей цели `bce + dice + 0.3*cls + 0.4*aux`:

1. **Маленькая маска тонет в BCE.** Правка на 0.5% кадра даёт 0.5% слагаемых,
   и градиент по ней теряется на фоне (армы L1, L2, L3).
2. **Цель не совпадает с метрикой.** AIC берёт Dice только по позитивам, а
   негативный кадр не штрафует вообще, пока его маска меньше 1% кадра. Dice же
   на негативе даёт почти полный штраф за любое пятно (армы L4, L5).

| арм | цель | что меняется относительно контроля |
| --- | --- | --- |
| L0 | `bce_dice` | контроль, формула не меняется |
| L1 | `balanced_bce_dice` | вес позитивных пикселей внутри кадра, до 20x |
| L2 | `focal_dice` | focal-перевес BCE, gamma=2, масштаб слагаемого сохранён |
| L3 | `focal_tversky` | Dice -> Tversky (alpha=0.3, beta=0.7, gamma=0.75) |
| L4 | `aic_surrogate` | Dice только по позитивам, негативы — мягкий FPR с порогом 1% |
| L5 | `aic_harmonic` | свёртка тоже как в метрике: гармоническое среднее по батчу |

Армы наследуют `baseline_protocol_originals.yaml`: независимый протокол
валидации (`dataset.protocol_path`), оригиналы в обучающей выборке
(`train_originals: true`) и валидация на исходном размере
(`eval.resolution: original`). От контроля каждый арм отличается только секцией
`loss:` — это проверяет `tests/test_ablation.py`.

Бюджет урезан вдвое, `epoch_size: 12000` вместо 24000 — 96 000 показов на арм.
Поэтому **сравнивать эти AIC с завершёнными полными прогонами нельзя**:
контроль L0 нужен именно как база на том же бюджете. Победивший арм переносится
в конфиг с полным бюджетом и новым `run_name`.

In [ ]:
import os
import sys
from dataclasses import replace
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
# dataset.protocol_path в конфигах — путь относительно корня проекта, а не
# относительно notebooks/, поэтому ядро работает из корня.
os.chdir(project_root)

from src.losses import list_losses
from src.training.ablation import LossAblation

list_losses()

## Выбор армов

Первым идёт L4: это главная гипотеза — цель, повторяющая форму метрики. Если
она не даст ничего против контроля, остальные четыре арма не стоят GPU-часов.
Контроль L0 идёт сразу за ним, потому что без него `delta_vs_control` не с чем
считать.

`data_path` берётся из конфигов; на другой машине путь до датасета
переопределяется здесь одной строкой, а не правкой шести файлов.

In [ ]:
ALL_ARMS = [
    "loss_l4_aic_surrogate",   # главная гипотеза, идёт первой
    "loss_l0_baseline",        # контроль, нужен для delta_vs_control
    "loss_l1_pos_weight",
    "loss_l2_focal",
    "loss_l3_focal_tversky",
    "loss_l5_aic_harmonic",
]

ARMS = ALL_ARMS[:2]   # L4 и контроль; ARMS = ALL_ARMS — вся серия

DATA_PATH = None      # например "/workspace/data" на удалённой машине
RESUME = True         # см. раздел про checkpoint ниже

ablation = LossAblation([project_root / "configs" / f"{arm}.yaml" for arm in ARMS],
                        data_path=DATA_PATH)
ablation.configs = [replace(config, train=replace(config.train, resume=RESUME))
                    for config in ablation.configs]

for config in ablation.configs:
    print(f"{config.paths.run_name:42s} {config.loss.name:18s} {config.loss.kwargs}")
print()
print("данные:   ", ablation.configs[0].paths.data_path)
print("протокол: ", ablation.configs[0].dataset.protocol_path)
print("оригиналы:", ablation.configs[0].dataset.train_originals)
print("валидация:", ablation.configs[0].eval.resolution)

## Расписание кадров

Обучение идёт в два этапа: сначала кадр режется на кропы (`crop_scale_range`, с
перевесом на кропы с разметкой — `foreground_crop_probability`), последние
`final_full_frame_epochs` эпох модель видит **только целые кадры**. Валидация
всё это время на исходном размере, поэтому последняя фаза убирает расхождение
между тем, на чём учились, и тем, на чём меряют.

Ячейка ниже печатает фактическое расписание — не то, что написано в YAML, а то,
что вернёт `AugmentationPipeline` на каждой эпохе.

In [ ]:
from src.data.augmentation.pipeline import AugmentationPipeline

config = ablation.configs[0]
pipeline = AugmentationPipeline(config.augmentation, total_epochs=config.train.epochs)

print(f"crop_scale_range={config.augmentation.crop_scale_range}, "
      f"foreground_crop_probability={config.augmentation.foreground_crop_probability}, "
      f"final_full_frame_epochs={config.augmentation.final_full_frame_epochs}")
print()
for epoch in range(config.train.epochs):
    pipeline.set_epoch(epoch)
    p = pipeline.full_frame_probability
    phase = "только целые кадры" if p == 1.0 else f"{1 - p:.0%} кропов / {p:.0%} целых кадров"
    print(f"эпоха {epoch + 1}/{config.train.epochs}:  p(full frame)={p:.2f}  {phase}")

## Проверка перед запуском

Один шаг на синтетическом батче: цель считается, градиент доходит до обеих
голов. Дешевле, чем узнать про опечатку в `loss.kwargs` через полчаса
обучения.

In [ ]:
import torch

from src.losses import build_loss

for config in ablation.configs:
    loss_fn = build_loss(config)
    logits = torch.randn(4, 1, 32, 32, requires_grad=True)
    mask = torch.zeros(4, 1, 32, 32)
    mask[:2, :, 2:6, 2:6] = 1.0  # два позитива с маской на 1.5% кадра, два негатива
    out = {"logits": logits, "aux_logits": logits, "cls_logits": torch.randn(4, 1)}
    batch = {"mask": mask, "label": (mask.flatten(1).sum(1, keepdim=True) > 0).float()}
    result = loss_fn(out, batch)
    result.total.backward()
    print(f"{config.loss.name:18s} loss={result.total.item():7.4f} |grad|={logits.grad.abs().sum():8.3f}")

## Продолжение с checkpoint

Каждая эпоха дописывает `runs/<run_name>/ckpt/last.pt` (модель, EMA, optimizer,
scheduler, scaler, номер эпохи), и, если AIC вырос, `ckpt/best.pt`. При
`RESUME = True` арм подхватывает `last.pt` и продолжает со следующей эпохи —
упавшее ядро или снятый под стоят ровно одну недосчитанную эпоху.

Два разных уровня пропуска, их легко перепутать:

* `RESUME` (в ячейке выбора армов) — продолжать ли **недосчитанный** арм с его
  `last.pt`. `RESUME = False` не затирает старую папку: `Run.create` уводит
  новый прогон в `<run_name>__MMDD-HHMMSS`, часы обучения не теряются никогда.
* `skip_completed` в `ablation.run()` — пропускать ли **досчитанные** армы,
  читая их сводку с диска. По умолчанию `True`, поэтому ячейку запуска можно
  перезапускать сколько угодно.

Продолжить с другой функцией потерь нельзя: `_check_resume_protocol` сверяет со
снапшотом `loss.name`, веса aux/DCT-голов, расписание кадров, протокол и
`train_originals` и падает с понятной ошибкой, вместо того чтобы дописать чужую
кривую обучения под тем же именем. Для новой конфигурации нужен новый
`run_name`.

Таблица ниже — что уже лежит на диске по каждому арму.

In [ ]:
import pandas as pd

from src.training.runs import Run

status = []
for config in ablation.configs:
    run_dir = config.paths.runs_path / config.paths.run_name
    run = Run.open(run_dir) if run_dir.is_dir() else None
    history = run.history if run is not None else pd.DataFrame()
    summary = run.summary if run is not None else {}
    status.append({
        "arm": config.paths.run_name.split("_loss_")[-1],
        "last.pt": (run_dir / "ckpt" / "last.pt").exists(),
        "best.pt": (run_dir / "ckpt" / "best.pt").exists(),
        "эпох в логе": len(history),
        "всего эпох": config.train.epochs,
        "лучший AIC": history["val/aic_tuned"].max() if "val/aic_tuned" in history else None,
        "досчитан": bool(summary.get("training_complete")),
        "папка": str(run_dir),
    })
pd.DataFrame(status)

Продолжить один конкретный арм, не трогая остальные, — отдельная ячейка на
случай, когда серия уже частично посчитана и нужен только один прогон:

```python
from src.training.engine import run_experiment

run_experiment(ablation.configs[0])   # арм из ARMS[0], продолжится с last.pt
```

Оценить чужой checkpoint, ничего не дообучая, — `src/eval/checkpoints.py`; он
читает `ckpt/best.pt` и `summary.json` того же прогона.

## Запуск серии

Армы идут подряд, в порядке `ARMS`. Упавший арм не останавливает серию —
причина попадёт в таблицу. Уже досчитанные прогоны пропускаются, поэтому ячейку
можно перезапускать после перезапуска ядра; чтобы пересчитать всё заново,
поставьте `skip_completed=False` и смените `run_name` в конфигах.

In [ ]:
results = ablation.run()
LossAblation.compare(results)

## Разбор

`delta_vs_control` — разница AIC с армом L0 на том же бюджете. Смотреть надо не
только на AIC: арм может выиграть Dice и проиграть FPR, и тогда его стоит
пробовать в паре с другим порогом `cls`, а не отбрасывать.

In [ ]:
table = LossAblation.compare(results)
table[["arm", "loss", "aic", "dice_pos", "fpr_neg", "mask_threshold", "cls_threshold"]]

In [ ]:
# Кривая обучения по армам: не выиграл ли арм просто за счёт более быстрого старта.
# Излом на последних final_full_frame_epochs эпохах — это переход на целые кадры,
# а не расхождение обучения.
history = pd.concat([
    Run.open(config.paths.runs_path / config.paths.run_name).history.assign(
        arm=config.paths.run_name.split("_loss_")[-1])
    for config in ablation.configs
    if (config.paths.runs_path / config.paths.run_name / "metrics.jsonl").exists()
])
history.pivot_table(index="epoch", columns="arm", values="val/aic_tuned")